# 04 — A complete tiny VLM from scratch (PyTorch)

We will train a miniature image-conditioned autoregressive language model. There are no pretrained weights, model wrappers, downloads, or hidden trainer. Images become patch tokens; word IDs become token embeddings; a causal decoder reads previous words while cross-attending to image tokens; next-token cross-entropy trains all parameters. This is a toy proof of mechanics only. Its tiny synthetic data, vocabulary, resolution, and compute do not resemble the scale or robustness of production VLM pretraining.

Diagram: RGB image -> patch projection -> visual memory; BOS + previous answer tokens -> token/position embeddings -> causal self-attention -> cross-attention to visual memory -> vocabulary logits. During training the target sequence is shifted one position. During inference we generate one token at a time.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

torch.manual_seed(31)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab = {'<pad>': 0, '<bos>': 1, '<eos>': 2, 'red': 3, 'blue': 4, 'block': 5, 'cup': 6}
id_to_word = {v: k for k, v in vocab.items()}
captions = {0: [1, 3, 5, 2], 1: [1, 4, 6, 2]}

def make_scene(label):
    image = torch.zeros(3, 16, 16)
    if label == 0:  # red block in upper-left patches
        image[0, 1:7, 1:7] = 1.0
    else:  # blue cup proxy in lower-right patches
        image[2, 9:15, 9:15] = 1.0
    image += 0.03 * torch.randn_like(image)
    return image.clamp(0, 1)

images, labels = [], []
for i in range(256):
    y = i % 2
    images.append(make_scene(y))
    labels.append(captions[y])
images = torch.stack(images).to(device)
tokens = torch.tensor(labels, dtype=torch.long, device=device)
perm = torch.randperm(len(images), device=device)
train_idx, test_idx = perm[:224], perm[224:]
print('device:', device, 'images:', tuple(images.shape), 'token sequences:', tuple(tokens.shape))

## Architecture

For readability, this uses PyTorch's MultiheadAttention primitive, but defines the complete network, objectives, and training loop directly. A production VLM would need a much stronger vision encoder, tokenizer, positional scheme, high-resolution handling, robust data mixture, and extensive training/evaluation.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(width, heads, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(width, heads, batch_first=True)
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(width), nn.LayerNorm(width), nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))

    def forward(self, x, visual_memory):
        length = x.shape[1]
        causal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool, device=x.device), diagonal=1)
        attended, _ = self.self_attn(x, x, x, attn_mask=causal_mask, need_weights=False)
        x = self.norm1(x + attended)
        attended, _ = self.cross_attn(x, visual_memory, visual_memory, need_weights=False)
        x = self.norm2(x + attended)
        return self.norm3(x + self.ff(x))

class TinyVLM(nn.Module):
    def __init__(self, vocab_size, width=48, max_text=8):
        super().__init__()
        self.vision = nn.Conv2d(3, width, kernel_size=4, stride=4)
        self.text = nn.Embedding(vocab_size, width)
        self.text_pos = nn.Parameter(torch.zeros(1, max_text, width))
        self.image_pos = nn.Parameter(torch.zeros(1, 16, width))
        self.block = DecoderBlock(width, heads=4)
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size, bias=False)
        nn.init.normal_(self.text_pos, std=0.02)
        nn.init.normal_(self.image_pos, std=0.02)

    def encode_image(self, images):
        features = self.vision(images).flatten(2).transpose(1, 2)
        return features + self.image_pos[:, :features.shape[1]]

    def forward(self, images, input_ids):
        memory = self.encode_image(images)
        x = self.text(input_ids) + self.text_pos[:, :input_ids.shape[1]]
        x = self.block(x, memory)
        return self.lm_head(self.final_norm(x))

model = TinyVLM(len(vocab)).to(device)
print('parameters:', sum(p.numel() for p in model.parameters()), 'visual memory:', tuple(model.encode_image(images[:2]).shape))

## Pretrain the toy model

Input is `[BOS, word1, word2]`; target is `[word1, word2, EOS]`. Every target token is supervised in this toy caption task. A real instruction dataset may mask user/prompt tokens using `-100`. AdamW is written as a direct optimizer use; no trainer wrapper. We keep a held-out split and report exact next-token accuracy as a toy diagnostic, not a benchmark.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
input_ids, targets = tokens[:, :-1], tokens[:, 1:]
for epoch in range(45):
    model.train()
    order = train_idx[torch.randperm(len(train_idx), device=device)]
    loss_sum = 0.0
    for ids in order.split(32):
        optimizer.zero_grad(set_to_none=True)
        logits = model(images[ids], input_ids[ids])
        loss = F.cross_entropy(logits.reshape(-1, len(vocab)), targets[ids].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += float(loss.detach()) * len(ids)
    if (epoch + 1) % 10 == 0:
        print(f'epoch {epoch + 1:02d} mean token loss {loss_sum / len(train_idx):.4f}')

model.eval()
with torch.no_grad():
    test_logits = model(images[test_idx], input_ids[test_idx])
    acc = (test_logits.argmax(-1) == targets[test_idx]).float().mean()
print('held-out next-token accuracy:', round(float(acc), 3))

## Autoregressive inference and inspection

Inference starts with BOS and repeatedly selects the next token from the last position. Greedy decoding is deterministic but not always best. Real generation also needs stop-token handling, max length, batching/caching, and safety constraints. Here, compare output words with the known synthetic class labels.

In [ ]:
@torch.no_grad()
def generate_one(model, image, max_new_tokens=4):
    model.eval()
    generated = torch.tensor([[vocab['<bos>']]], device=image.device)
    for _ in range(max_new_tokens):
        next_logits = model(image.unsqueeze(0), generated)[:, -1, :]
        next_id = next_logits.argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        if int(next_id.item()) == vocab['<eos>']:
            break
    return ' '.join(id_to_word[int(i)] for i in generated[0, 1:] if int(i) != vocab['<eos>'])

rows = []
for idx in test_idx[:6].tolist():
    rows.append({'expected': 'red block' if int(tokens[idx, 1]) == vocab['red'] else 'blue cup',
                 'generated': generate_one(model, images[idx])})
pd.DataFrame(rows)

## Experiments and limitations

Try: remove cross-attention and see whether image-conditioned accuracy collapses; shuffle image-caption pairs; change patch size/resolution; freeze vision weights; compare train and held-out loss; add a third synthetic class; use label smoothing; save/reload `state_dict`. The synthetic task is intentionally easy and contains only two visual patterns. It cannot teach open-world recognition, grounding, OCR, reasoning, or robust robot control. A meaningful pretraining effort needs large, rights-cleared image-text data and rigorous evaluation.